# Anchor-free

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/12-anchor-free/)

先執行環境格，再閱讀、執行下方完整實驗。各節互相獨立，不需要上一節的 runtime。

本版使用 CPU 驗證機制及短步參數更新；長訓練的辨識效果、AP 與 GPU 效率仍待補測。這些範例沒有預訓練權重，也不需下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.1.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 本輪只驗證 CPU。已裝相同 PyTorch 版本時保留其 CPU/CUDA build。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
"""Point + four distances: a simplified anchor-free regression experiment."""
import torch
from torch import nn
from torch.nn import functional as F


def decode(points, distances, stride):
    left_top, right_bottom = distances[..., :2], distances[..., 2:]
    return torch.cat((points - left_top * stride, points + right_bottom * stride), -1)


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    point = torch.tensor([[24., 24.]])  # pixel x,y; stride is pixels per feature cell
    gt = torch.tensor([[12., 16., 40., 36.]])
    distance = torch.cat((point - gt[:, :2], gt[:, 2:] - point), -1) / 8
    assert torch.allclose(distance, torch.tensor([[1.5, 1., 2., 1.5]]))
    assert torch.allclose(decode(point, distance, 8), gt)
    # This head predicts positive continuous distances, not anchor width/height offsets.
    raw = nn.Parameter(torch.zeros(1, 4))
    optimizer = torch.optim.SGD([raw], lr=.5)
    before = F.smooth_l1_loss(F.softplus(raw), distance).item()
    for _ in range(80):
        optimizer.zero_grad()
        loss = F.smooth_l1_loss(F.softplus(raw), distance)
        loss.backward()
        assert torch.isfinite(raw.grad).all()
        optimizer.step()
    after = F.smooth_l1_loss(F.softplus(raw), distance).item()
    assert after < before / 100
    print('target ltrb in feature cells:', distance.tolist())
    print('decoded target pixels:', decode(point, distance, 8).tolist())
    print(f'distance loss {before:.6f} -> {after:.6f}')
    print('learned box pixels:', decode(point, F.softplus(raw).detach(), 8).round(decimals=2).tolist())
    # A point outside the target cannot represent that box with all-positive distances.
    outside = torch.tensor([[48., 24.]])
    assert (torch.cat((outside - gt[:, :2], gt[:, 2:] - outside), -1) < 0).any()
    print('outside point requires a negative distance: assignment is still necessary')


if __name__ == '__main__':
    main()


target ltrb in feature cells: [[1.5, 1.0, 2.0, 1.5]]
decoded target pixels: [[12.0, 16.0, 40.0, 36.0]]
distance loss 0.376236 -> 0.000012
learned box pixels: [[12.029999732971191, 16.059999465942383, 39.9900016784668, 35.970001220703125]]
outside point requires a negative distance: assignment is still necessary
